# MobileRec: Chronological Data Split

This notebook splits the prepared interaction dataset into training, validation, and test sets. It uses all users and their complete histories from the final `Parquet` file produced in the previous notebook.

In [1]:
from pathlib import Path
from time import perf_counter
import duckdb

In [2]:
run_name = "all_users"
seed = 42

The random seed is fixed at `seed = 42` to support reproducibility in subsequent neural model training.

In [3]:
final_path = Path("artifacts") / "final" / "interactions_final.parquet"
split_dir = Path("artifacts") / "split_all_users"
temp_dir = Path("artifacts") / "split_temp"

split_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(exist_ok=True)

if not final_path.exists():
    raise FileNotFoundError(f"File does not exist: {final_path}")

final_path_sql = final_path.as_posix()
temp_path_sql = temp_dir.as_posix()

print(final_path_sql)
print(temp_path_sql)

artifacts/final/interactions_final.parquet
artifacts/split_temp


The `final_path` variable points to the processed interaction dataset in `Parquet` format. Split files are saved in `split_dir`, while `temp_dir` stores temporary DuckDB files used during processing.

In [4]:
con = duckdb.connect()

user_check = con.execute(f"""SELECT
    COUNT(DISTINCT uid) AS users,
    COUNT(*) AS interactions
    FROM read_parquet('{final_path_sql}')""").fetchdf()

con.close()

user_check

,users,interactions
0,700111,19297019


In [5]:
con = duckdb.connect()

history_check = con.execute(f"""SELECT
    MIN(interaction_count) AS minimum,
    AVG(interaction_count) AS average,
    MEDIAN(interaction_count) AS median,
    MAX(interaction_count) AS maximum
    FROM (
        SELECT uid,
        COUNT(*) AS interaction_count
        FROM read_parquet('{final_path_sql}')
        GROUP BY uid)""").fetchdf()

con.close()

history_check

,minimum,average,median,maximum
0,5,27.562799,8.0,253


The dataset is confirmed to contain all 700,111 users and 19,297,019 interactions. User history lengths range from five to 253 interactions, with a median of eight. No users are excluded based on history length.

In [6]:
history_path = split_dir / "complete_history_all_users.parquet"

if history_path.exists():
    print("Complete histories already exist:", history_path)
else:
    start_time = perf_counter()
    con = duckdb.connect()
    con.execute("SET memory_limit = '2GB'")
    con.execute(f"SET temp_directory = '{temp_path_sql}'")
    con.execute(f"""COPY (WITH source_rows AS (
        SELECT ROW_NUMBER() OVER () AS source_row_order, uid, app_package, formated_date, unix_timestamp, rating
        FROM read_parquet('{final_path_sql}'))
        SELECT uid, app_package, formated_date, unix_timestamp, rating, source_row_order
        FROM source_rows
        ORDER BY source_row_order) TO '{history_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.close()
    print(f"Histories were saved in {(perf_counter()-start_time)/60:.2f} minutes")

Complete histories already exist: artifacts\split_all_users\complete_history_all_users.parquet


In [7]:
con = duckdb.connect()

history_sum = con.execute(f"""SELECT
    (SELECT COUNT(*) FROM read_parquet('{final_path_sql}')) AS expected_rows,
    (SELECT COUNT(*) FROM read_parquet('{history_path.as_posix()}')) AS saved_rows,
    (SELECT COUNT(DISTINCT uid) FROM read_parquet('{history_path.as_posix()}')) AS saved_users""").fetchdf()

con.close()

if history_sum.loc[0, "expected_rows"] != history_sum.loc[0, "saved_rows"]:
    raise ValueError("Complete histories have not been saved.")

history_sum

,expected_rows,saved_rows,saved_users
0,19297019,19297019,700111


Complete user histories are saved before splitting. The resulting file contains all 19,297,019 interactions and 700,111 users, together with the original row order used to resolve ties when multiple interactions share the same date.

In [8]:
train_path = split_dir / "train_all_users.parquet"
valid_path = split_dir / "validation_all_users.parquet"
test_path = split_dir / "test_all_users.parquet"

split_paths = [train_path, valid_path, test_path]

if all(path.exists() for path in split_paths):
    print("Train, validation, and test files already exist.")
elif any(path.exists() for path in split_paths):
    raise FileExistsError("Only part of the split files exists.")
else:
    start_time = perf_counter()
    con = duckdb.connect()
    con.execute("SET memory_limit = '2GB'")
    con.execute(f"SET temp_directory = '{temp_path_sql}'")
    con.execute(f"""CREATE TEMP TABLE ranked_data AS SELECT uid, app_package, formated_date, unix_timestamp, rating, source_row_order,
        ROW_NUMBER() OVER (PARTITION BY uid 
        ORDER BY unix_timestamp, source_row_order) AS event_position,
        COUNT(*) OVER (PARTITION BY uid) AS user_events
        FROM read_parquet('{history_path.as_posix()}')""")
    con.execute(f"""COPY (
        SELECT uid, app_package, formated_date, unix_timestamp, rating, source_row_order
        FROM ranked_data
        WHERE event_position <= user_events - 2
        ORDER BY uid, event_position) TO '{train_path.as_posix()}'(FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.execute(f"""COPY (
        SELECT uid, app_package, formated_date, unix_timestamp, rating, source_row_order
        FROM ranked_data
        WHERE event_position = user_events - 1
        ORDER BY uid) TO '{valid_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.execute(f"""COPY (
        SELECT uid, app_package, formated_date, unix_timestamp, rating, source_row_order
        FROM ranked_data
        WHERE event_position = user_events
        ORDER BY uid) TO '{test_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.close()
    print(f"Split je napravljen za {(perf_counter()-start_time)/60:.2f} minutes")


Train, validation, and test files already exist.


Each user’s interactions are ordered from earliest to latest by `unix_timestamp`. When multiple interactions occur on the same date, `source_row_order`, their position in the original file, determines their order.

All but the last two interactions are assigned to the training set. The second-to-last interaction is the validation target, and the final interaction is the test target. This split prevents a user’s later interaction from entering the training data when an earlier interaction is the prediction target.

In [9]:
con = duckdb.connect()
split_sum = con.execute(f"""SELECT
    (SELECT COUNT(*) FROM read_parquet('{history_path.as_posix()}')) AS all_rows,
    (SELECT COUNT(*) FROM read_parquet('{train_path.as_posix()}')) AS train_rows,
    (SELECT COUNT(*) FROM read_parquet('{valid_path.as_posix()}')) AS validation_rows,
    (SELECT COUNT(*) FROM read_parquet('{test_path.as_posix()}')) AS test_rows,
    (SELECT COUNT(DISTINCT uid) FROM read_parquet('{train_path.as_posix()}')) AS train_users,
    (SELECT COUNT(DISTINCT uid) FROM read_parquet('{valid_path.as_posix()}')) AS validation_users,
    (SELECT COUNT(DISTINCT uid) FROM read_parquet('{test_path.as_posix()}')) AS test_users""").fetchdf()
con.close()

split_rows = (split_sum.loc[0, "train_rows"]+ split_sum.loc[0, "validation_rows"]+ split_sum.loc[0, "test_rows"])

split_sum

,all_rows,train_rows,validation_rows,test_rows,train_users,validation_users,test_users
0,19297019,17896797,700111,700111,700111,700111,700111


The split produces 17,896,797 training interactions and 700,111 target interactions in each of the validation and test sets. Every user appears in all three sets, and the total row count matches the complete interaction history.

In [10]:
user_map_path = split_dir / "users_map_all_users.parquet"
item_map_path = split_dir / "items_map_all_users.parquet"
pop_path = split_dir / "apps_popularity_all_users.parquet"
coverage_path = split_dir / "start_coverage_all_users.parquet"

map_paths = [user_map_path, item_map_path, pop_path, coverage_path]

if all(path.exists() for path in map_paths):
    print("Mappings and popularity statistics already exist.")
elif any(path.exists() for path in map_paths):
    raise FileExistsError("Only part of the mapping files exists.")
else:
    start_time = perf_counter()
    con = duckdb.connect()
    con.execute("SET memory_limit = '2GB'")
    con.execute(f"SET temp_directory = '{temp_path_sql}'")
    con.execute(f"""CREATE TEMP TABLE train_data AS
                    SELECT uid, app_package
                    FROM read_parquet('{train_path.as_posix()}');
                    
                    CREATE TEMP TABLE user_map AS
                    SELECT uid,
                    ROW_NUMBER() OVER (ORDER BY uid) - 1 AS user_idx
                    FROM (SELECT DISTINCT uid FROM train_data);
                    
                    CREATE TEMP TABLE item_map AS
                    SELECT app_package,
                    ROW_NUMBER() OVER (ORDER BY app_package) - 1 AS item_idx
                    FROM (SELECT DISTINCT app_package FROM train_data);

                    CREATE TEMP TABLE app_popularity AS
                    SELECT item_map.item_idx, train_data.app_package,
                    COUNT(*) AS interaction_count,
                    COUNT(DISTINCT uid) AS user_count
                    FROM train_data
                    INNER JOIN item_map USING (app_package)
                    GROUP BY item_map.item_idx, train_data.app_package
                    ORDER BY interaction_count DESC, train_data.app_package;

                    CREATE TEMP TABLE coverage AS
                    WITH targets AS (SELECT 'validation' AS split_name, app_package
                    FROM read_parquet('{valid_path.as_posix()}')
                    UNION ALL
                    SELECT 'test' AS split_name, app_package
                    FROM read_parquet('{test_path.as_posix()}'))
                    SELECT split_name,
                    COUNT(*) AS all_targets,
                    SUM(CASE WHEN item_map.app_package IS NOT NULL THEN 1 ELSE 0 END) AS known_items,
                    SUM(CASE WHEN item_map.app_package IS NULL THEN 1 ELSE 0 END) AS cold_items,
                    100.0 * SUM(CASE WHEN item_map.app_package IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*) AS coverage_percent
                    FROM targets
                    LEFT JOIN item_map USING (app_package)
                    GROUP BY split_name
                    ORDER BY split_name;""")

    for table, path, col in [
    ("user_map", user_map_path, "user_idx"),
    ("item_map", item_map_path, "item_idx"),
    ("app_popularity", pop_path, "interaction_count DESC, app_package"),
    ("coverage", coverage_path, "split_name")]:
        con.execute(f"""COPY (SELECT * FROM {table} ORDER BY {col}) TO '{path.as_posix()}'(FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.close()
    print(f"Mapiranja su napravljena za {(perf_counter()-start_time)/60:.2f} minutes")


Mappings and popularity statistics already exist.


In [11]:
con = duckdb.connect()
model_sum = con.execute(f"""SELECT
                            (SELECT COUNT(*) FROM read_parquet('{train_path.as_posix()}')) AS train_rows,
                            (SELECT COUNT(*) FROM read_parquet('{user_map_path.as_posix()}')) AS users,
                            (SELECT COUNT(*) FROM read_parquet('{item_map_path.as_posix()}')) AS candidate_apps""").fetchdf()

coverage_sum = con.execute(f""" SELECT *
                                FROM read_parquet('{coverage_path.as_posix()}')
                                ORDER BY split_name""").fetchdf()
con.close()

model_sum

,train_rows,users,candidate_apps
0,17896797,700111,10172


In [12]:
coverage_sum

,split_name,all_targets,known_items,cold_items,coverage_percent
0,test,700111,700096.0,15.0,99.997857
1,validation,700111,700108.0,3.0,99.999571


Numeric user and app mappings, app popularity, and the candidate set are derived exclusively from training interactions. The resulting mappings contain 700,111 users and 10,172 candidate apps.

Three of the 700,111 validation targets and 15 of the 700,111 test targets refer to an app absent from training. These item cold-start targets cannot be recommended by models based only on training interactions and are excluded from the standard ranking metrics. Candidate-set coverage is 99.9996% for validation and 99.9979% for testing.